# Weather API + Join

### Imports

In [4]:
import pandas as pd 
import requests 
import time

### Weather API Function

In [5]:
BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def weather_retrieval(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max,pressure_msl_mean',
        'timezone': 'America/New_York',
    }

    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame 

FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=30),
    'precipitation_sum': [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    'windspeed_10m_max': [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                          15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                          19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    'pressure_msl_mean': [1014.1, 1014.1, 1015.7, 1017.8, 1017.3, 1014.3, 1011.9, 1013.8, 
                          1015.7, 1014.3, 1012.3, 1011.8, 1012.7, 1012.2, 1011.8, 1012.9, 1013.7, 
                          1012.9, 1011.2, 1010.6, 1012.5, 1012.7, 1013.2, 1013.5, 1011.7, 1004.1, 1006.7, 1012.3, 1013.1, 1013.8],
    'source': 'fallback sample',
})

try:
    weather_live = weather_retrieval('2024-09-01', '2024-09-30')
    print('Weather retrieval succeeded.')
except requests.RequestException as err:
    print('API call failed:', err)
    weather_live = FALLBACK

Weather retrieval succeeded.


### Weather/Sales Join Function

In [6]:
def weather_sales_join(sales_data, weather_data):
    sales_data = sales_data.copy() 
    sales_data['date'] = sales_data['timestamp'].dt.normalize()

    merged = sales_data.merge(weather_data, on='date', how='left', indicator=True, validate='many_to_one')
    unmatched = merged.loc[merged['_merge'] == 'left_only', 'date'].drop_duplicates()

    print(merged['_merge'].value_counts())
    print('Number of sales days with no weather:', len(unmatched))
    if(len(unmatched) > 0):
        print('Sales days with no weather:', unmatched.dt.date.tolist())
    print('Rows kept by the left join:', len(merged), 'of', len(sales_data))

    assert len(merged) == len(sales_data), 'The join changed the row count.'
    assert merged['quantity'].sum() == sales_data['quantity'].sum(), 'The quantity total changed.'
    assert len(unmatched) == 0, 'There are sales days with no weather.'
    print('All checks passed.')
    
    return merged